In [ ]:
import os
import sys

IS_COLAB = "google.colab" in sys.modules
IS_DATABRICKS = "DATABRICKS_RUNTIME_VERSION" in os.environ

if IS_COLAB:
    print("Colab environment. Make sure the HW_DIR below matches what you used in the startup notebook.")
    print("If you are asked to restart the runtime, you do not need to do so.")
    print("If you decide to restart the runtime, you will need to re-run this cell.")
    from google.colab import drive
    drive.mount("/content/drive")
    HW_DIR = "/content/drive/MyDrive/CS189/hw/hw4-pt1"
    os.chdir(HW_DIR)
    %pip install -r colab_requirements.txt
elif IS_DATABRICKS:
    print("Databricks environment. Make sure your base environment is AI v6.")
    %pip install -r databricks_requirements.txt
else:
    print("Local development. Make sure your kernel has everything in requirements.txt.")

if not IS_COLAB and not IS_DATABRICKS:
    print("Local development is discouraged due to need for accelerators.")


In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("hw4_pt_1.ipynb")

<h1 class="cal cal-h1">Homework 4 Part 1 – CNNs and Transformers</h1>

Welcome to Homework 4 Part 1! You will implement foundational CNN and transformer building blocks in PyTorch.

---

## Due Date: Friday, November 13, 11:59 PM

This assignment is due on **Friday, November 13 at 11:59 PM**. You must submit your work to the programming assignment by this deadline. Please refer to the syllabus for the [Slip Day policy](https://eecs189.org/fa26/syllabus/#slip-days).

### Submission Tips:
- **Plan ahead**: Submit several hours before the deadline.
- **Reach out for help early**: Contact course staff well before the deadline.
- **Use a GPU runtime** on Colab (T4) or Databricks (A10) for training cells.

---

### Grading Breakdown

| Question | Manual Grading? | Points |
|----------|-----------------|--------|
| 1a       | No              | 2      |
| 1bi      | No              | 1      |
| 1bii     | Yes             | 2      |
| 2a       | No              | 2      |
| 2b       | No              | 3      |
| 2ci      | No              | 1      |
| 2cii     | Yes             | 1      |
| 3a       | No              | 1      |
| 3b       | No              | 2      |
| 3c       | No              | 2      |
| 3d       | No              | 2      |
| 3e       | No              | 3      |
| 3f       | No              | 3      |
| 3g       | No              | 3      |
| 3hi      | No              | 1      |
| 3hii     | Yes             | 1      |
| 3i       | No              | 1      |
| **Total**|                 | **31** |

**Note**: Manual questions are figures saved under `written/` and uploaded as a PDF to the associated paper assignment.

---

### Autograding
- After each coding question, run `grader.check("q...")` to execute the **public** tests.
- Keep the `tests/` folder next to this notebook.
- Submit the exported programming zip. Written figures must be under `written/`.

Good luck!


In [ ]:
TESTS_VERSION = "v1"
REPULL_TESTS = False

if REPULL_TESTS:
    !git fetch origin tag {TESTS_VERSION}
    !git restore --source={TESTS_VERSION} -- tests/


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
import random
from datasets import load_dataset, load_from_disk
import plotly
import plotly.subplots
import plotly.graph_objects
import os
import json
import re
from tqdm import tqdm
from torch.utils.data import DataLoader, Dataset, TensorDataset
import torchvision.transforms as transforms


### **IMPORTANT:**
- Do not change the random seed values!!!
- Make sure to turn on the GPU accelerator in the runtime settings before starting the notebook.


In [ ]:
def set_seed(seed):
    """
    Set the seed for all random number generators
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

# Set random seeds for reproducible results
SEED = 189
set_seed(SEED)

IS_GRADING_ENV = os.getenv("IS_GRADING_ENV") == "true"

if IS_GRADING_ENV:
    device = "cpu"
elif torch.accelerator.is_available():
    device = torch.accelerator.current_accelerator()
else:
    device = "cpu"

print(f"Using device: {device}") # Check that the current device is cuda or another GPU accelerator!


In [ ]:
import json

WRITTEN_DIR = "written"
os.makedirs(WRITTEN_DIR, exist_ok=True)

# Shipped in tests/ by build_release.py from PARTS[n]["written_files"].
with open(os.path.join("tests", "written_files.json"), encoding="utf-8") as f:
    EXPECTED_WRITTEN_FILES = json.load(f)


def save_figure(fig, question_name):
    """Save a Plotly or Matplotlib figure under written/."""
    if IS_GRADING_ENV:
        return None
    path = os.path.join(WRITTEN_DIR, f"{question_name}.png")
    if fig is None:
        raise ValueError(
            f"Cannot save {path}: the figure for {question_name} is None. "
            "Check that the cell or function building this figure returns it."
        )
    if hasattr(fig, "write_image"):
        fig.write_image(path)
    elif hasattr(fig, "savefig"):
        fig.savefig(path, bbox_inches="tight")
    else:
        raise TypeError(
            f"Cannot save {path}: expected a Plotly or Matplotlib figure for "
            f"{question_name}, but got {type(fig).__name__}."
        )
    print(f"Saved {path}")
    return path


def save_written_answer(answer, question_name):
    """Save a written response under written/."""
    if IS_GRADING_ENV:
        return None
    path = os.path.join(WRITTEN_DIR, f"{question_name}.txt")
    try:
        with open(path, "w", encoding="utf-8") as f:
            f.write(str(answer).strip() + "\n")
    except Exception as e:
        print(f"WARNING: Could not save {path}: {e}")
        return None
    print(f"Saved {path}")
    return path


def warn_missing_written_files():
    """Print which expected written/ files are missing locally. Does not fail the cell."""
    missing = [
        name
        for name in EXPECTED_WRITTEN_FILES
        if not os.path.isfile(os.path.join(WRITTEN_DIR, name))
    ]
    if missing:
        print("Missing written files:", ", ".join(missing))
    else:
        print(f"All {len(EXPECTED_WRITTEN_FILES)} expected files found in written/.")


def plot_metrics(train_losses, val_losses, train_accuracies=None, val_accuracies=None,
                 num_epochs=None, title=""):
    """Return one figure containing loss curves and, when provided, accuracy curves."""
    metrics = [(train_losses, "Training Loss"), (val_losses, "Validation Loss")]
    if train_accuracies is not None and val_accuracies is not None:
        metrics += [(train_accuracies, "Training Accuracy"),
                    (val_accuracies, "Validation Accuracy")]
    epochs = list(range(1, (num_epochs or len(train_losses)) + 1))
    titles = [f"{label} (final: {values[-1]:.3f})" if values else label
              for values, label in metrics]
    fig = plotly.subplots.make_subplots(rows=len(metrics) // 2, cols=2,
                                       subplot_titles=titles)
    for i, (values, label) in enumerate(metrics):
        row, col = i // 2 + 1, i % 2 + 1
        fig.add_trace(plotly.graph_objects.Scatter(x=epochs, y=values, mode="lines+markers",
                                                   name=label), row=row, col=col)
        fig.update_xaxes(title_text="Epoch", row=row, col=col)
        fig.update_yaxes(title_text=label.split()[-1], row=row, col=col)
    fig.update_layout(title=title, showlegend=False)
    return fig


# Question 1: Convolutional Neural Networks

First, let's warm up by building a CNN (Convolutional Neural Network) in PyTorch to get familiar with the PyTorch library.

### So what is PyTorch?

PyTorch is a powerful Python library for building and training neural networks. It is built around data structures called tensors (multi-dimensional arrays!). PyTorch is awesome because it handles creating computational graphs and calculating gradients for us.

#### PyTorch Modules

PyTorch's main building blocks are *modules*, which you can think of as different layers in your neural net. PyTorch has a couple of pre-defined classes which inherit from the base `nn.Module` class.

Modules can be as simple as individual layers:
* [Linear fully-connected layer](https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html)

... activation functions:
* [ReLU](https://docs.pytorch.org/docs/stable/generated/torch.nn.ReLU.html)

... entire blocks of a model, which often have *other* modules inside of them!
* [LSTM/long short-term memory blocks](https://docs.pytorch.org/docs/stable/generated/torch.nn.LSTM.html)

... or complete models!
* [Transformer - we'll be implementing our own version later in this homework](https://docs.pytorch.org/docs/stable/generated/torch.nn.Transformer.html)

In short, you can think of modules as LEGO bricks that you can piece together in different ways.

#### Implementing custom `nn.Module` subclasses!

We can also define our own classes that inherit from the `nn.Module` base class! To implement your own subclass of `nn.Module`, you need to define 2 methods:
* `__init__(self, ...)`: The constructor method where you initialize all the layers and parameters your module will use. Call super().__init__() first to properly initialize the parent nn.Module class. Define any layers (e.g., Conv2d, Linear, etc.) as instance attributes here. PyTorch will automatically track these as submodules and register their parameters.

* `forward(self, x)`: Defines the forward pass computation. Takes input tensor(s) and returns output tensor(s). This is where you specify how data flows through your layers—apply operations, activations, skip connections, etc. You call your model with output = model(input), which internally invokes the forward method. PyTorch's autograd automatically builds the backward pass from your forward operations.

#### PyTorch Functionals

Another important part of PyTorch is `torch.nn.functional`. This is a module (note: "module" in this case refers to a sublibrary of a Python library. It's not the same thing as the `Module` classes we were just talking about! I know, it's confusing!) of stateless functions, which means that these functions perform operations on tensors without maintaining any state or tracking parameters. So these don't contain learnable parameters we can train and update. `torch.nn.functional` is often imported as `F`; in practice you'll often see it being called like this: `F.relu(x)`.

Often times, certain kinds of layers have both `nn.Module` and `nn.functional` implementations in PyTorch. How do you know when to use which kind? As a rule of thumb, whenever you need the layer's parameters to be learnable (i.e. changeable over the course of training), use the `nn.Module` version. On the other hand, for operations like activation functions (e.g. ReLU or tanh) we don't have any parameters that we can learn! So their `nn.Module` and `nn.functional` version can be used interchangeably. In your code below, you can choose to use either `nn.ReLU` or `F.relu`.

| Aspect   | nn.Module | nn.Functional |
|----------|-----------------|---------------|
| State | Stateful - maintains parameters | Stateless - just functions|
| Parameters | Automatically registers learnable parameters and tracks them for optimization | No parameter tracking |
| Implementation details | Need to initialize the module first (e.g. using `self.relu = nn.ReLU()`) before using it (e.g. `a = self.relu(x)`) | Can call it directly on an input (e.g. `a = F.relu(x)`) |
| Usage examples | `nn.Linear(10, 5), nn.ReLU()` | `F.relu(x)` |


## Question 1a

Using some of PyTorch's existing `nn.Module` classes, we can build our own neural networks. Let's try building a CNN with the following architecture:

> * **Convolution Layer 1**: 16 output channels, kernel size 3 x 3, stride 2, padding 1
> * **ReLU Activation**: Element-wise ReLU
> * **Convolution Layer 2**: 16 output channels, kernel size 7 x 7, stride 2, padding 1
> * **ReLU Activation**: Element-wise ReLU
> * **Linear Classifier**: Linear layer for classification

For this question, assume inputs have shape `(batch_size, 3, 224, 224)`.

**Task:**
1. Implement the `__init__` method.
    1. Don't forget to call `super().__init__()`!
    2. Instantiate all the layers we'll need for our CNN.
2. Implement the `forward` method.
    1. Pass the input `x` through each of the layers.
    2. Return the final output.

**Required attribute names:** our tests load fixed weights into `self.conv1`, `self.conv2`, and `self.linear` before calling `forward`. Use exactly these names, and assign them to the `Conv2d` / `Linear` modules themselves (do not wrap ReLU inside `nn.Sequential` under those names).

**Hints:**
* The documentation for the `Conv2d` class [here](https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv2d.html) will contain helpful information about what parameters you need to specify to instantiate a `Conv2d` layer
* Since our images have 3 channels for Red, Green, and Blue, how many input channels should our first `Conv2d` layer have?
* How many channels does the output of the 1st ReLU layer have? What should be the number of input channels for the 2nd `Conv2d` layer?
* The final linear layer expects a flattened tensor for each item in the batch. What shape is the output after the 2nd ReLU? (Hint: you can either calculate the shape explicitly using formulas for what the dimensions of a conv layer's feature map will be, or add a print statement!) How can you flatten it into a 2D tensor with shape `(batch_size, num_features)`, where `num_features` is the product of the remaining channels, height, and width?
    * Helpful website to calculate the output shapes after a convolution: https://madebyollin.github.io/convnet-calculator/
* The linear classifier will make predictions about which class the image belongs to by outputting a vector of **logits** of length `num_classes` (one score per class) for each image in the batch. Do **not** apply softmax here — `CrossEntropyLoss` expects raw logits. What should be the output dimension for the linear classifier?
* You can choose to either instantiate a `nn.ReLU` layer in `__init__`, or directly call the functional `F.relu` in your forward pass!

**Helpful Documentation**
* [`nn.Conv2d`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv2d.html)
* [`nn.Linear`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html)
* [`nn.ReLU`](https://docs.pytorch.org/docs/stable/generated/torch.nn.ReLU.html)
* [`F.relu`](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.relu.html)


In [ ]:
class CNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        # TODO: Instantiate the layers your CNN will have.
        # Tests require these names:
        self.conv1 = ...
        self.conv2 = ...
        self.linear = ...
        # Optional: You can also define your ReLU layers here if you'd like
        # Otherwise, if you decide to use F.relu you can just remove these lines
        # self.relu1 = ...
        # self.relu2 = ...

    def forward(self, x):
        # TODO: 1. Pass the input through the first conv layer
        conv1_out = ...

        # TODO: 2. Apply ReLU
        relu1_out = ...

        # TODO: 3. Pass to the 2nd conv layer
        conv2_out = ...

        # TODO: 4. Apply ReLU
        relu2_out = ...

        # TODO: 5. Flatten before linear layer
        flat = ...

        # TODO: 6. Pass to linear layer
        out = ...

        return out


In [ ]:
grader.check("q1a")

### Check: CNN output shape

A helpful debugging tip while building your own deep-learning models or custom modules is to check the shapes of your outputs!

Run this cell **after** you have implemented `CNN.forward` (the stub will raise an error).

Below, we create a tensor of shape `(1, 3, 224, 224)` (batch, channels, height, width) filled with random numbers between [0, 1] and pass it through our `CNN` to check if the output has the shape we expect.

How do we determine what the shape of our output should be? We can trace through what the shapes should be after each of our layers.

Assume we instantiate a `CNN` with `num_classes=10` and we pass an input of shape `(1, 3, 224, 224)` through the `CNN`.

These would be the shapes after each layer:
1. Input: 1 × 3 × 224 × 224 --> dimensions: Batch size × 3 RGB channels × height of 224 pixels × width of 224 pixels
2. After conv1: 1 × 16 × 112 × 112 --> The first convolution produces feature maps with shape 112 × 112 and increases the number of channels to 16
3. After the first ReLU: 1 × 16 × 112 × 112 --> ReLU doesn't change the output dimensions
4. After conv2: 1 × 16 × 54 × 54 --> The second convolution produces feature maps with shape 54 × 54. This convolution doesn't change the number of channels
5. After the 2nd ReLU: 1 × 16 × 54 × 54 --> ReLU doesn't change the output dimensions
6. After the linear layer: 1 × 10 --> A tensor containing the logits for each class

You should verify that the shape of your actual output is also 1 x 10!


In [ ]:
x = torch.rand((1, 3, 224, 224))  # 1 image, 3 RGB channels, 224 x 224 pixels
print(f"Shape of input: {x.shape}")

res = CNN(num_classes=10)
out = res(x)
print(f"Shape of output: {out.shape}")
assert out.shape == torch.Size([1, 10]), f"Expected output shape (1, 10), but got {out.shape}"


## Load the Data

Before we jump into training our models, let's discuss how to work with data in our training loop.

For this assignment we use a 10-class subset of Mini-ImageNet, a small version of ImageNet-1k published by timm (PyTorch Image Models). We've zipped the subset for you at `data/mini_imagenet_10c.zip`:

* 1,000 training images
* 200 validation images
* classes 0 through 9

That Mini-ImageNet release is an ImageNet derivative under ImageNet's non-commercial research terms.

**Notes:**
* We're only using 1000 training images and 200 validation images for learning purposes. In a real-world scenario, you'd likely use a far larger dataset for training and evaluation :)

HuggingFace datasets behave like lists of dictionaries. Each Mini ImageNet sample has an `image` (a PIL Image, typically RGB with pixel values in 0–255) and a `label` (an integer class ID). The mapping from those IDs to readable names lives in `data/imagenet_class_id_to_name.json`.


In [ ]:
# Load a dictionary that maps numeric class IDs to string class names
with open('data/imagenet_class_id_to_name.json', 'r') as file:
    class_id_to_name = json.load(file)
    class_id_to_name = {int(key): value for key, value in class_id_to_name.items()}
    print(f"All classes in the mini-ImageNet label map: {class_id_to_name}")

if IS_GRADING_ENV:
    from PIL import Image
    _img = Image.fromarray(np.zeros((224, 224, 3), dtype=np.uint8))
    mini_imagenet_train = [{"image": _img, "label": i % 10} for i in range(1000)]
    mini_imagenet_val = [{"image": _img, "label": i % 10} for i in range(200)]
else:
    import zipfile
    subset_zip = "data/mini_imagenet_10c.zip"
    subset_dir = "data/mini_imagenet_10c"
    if not os.path.isdir(subset_dir):
        with zipfile.ZipFile(subset_zip) as zf:
            zf.extractall("data")
    mini_imagenet = load_from_disk(subset_dir)
    mini_imagenet_train = mini_imagenet["train"]
    mini_imagenet_val = mini_imagenet["validation"]
    print(mini_imagenet_train)
    print(mini_imagenet_val)


### PyTorch Datasets and DataLoaders

The data pipeline is provided for you so you can focus on defining and training the neural networks. It's still worth reading through: this is code you'll write often.

PyTorch's `Dataset` stores samples and labels. A `DataLoader` wraps an iterable around it so a training loop can pull shuffled batches. There is a short tutorial [here](https://docs.pytorch.org/tutorials/beginner/basics/data_tutorial.html).

`MiniImageNetDataset` below is a thin wrapper around the HuggingFace split: `__getitem__` returns a float image tensor and a long label. Two conventions matter later in the training loop:
* Model inputs should be `dtype=torch.float`; classification labels should be `dtype=torch.long`.
* Images are often stored as (H, W, C), but PyTorch models expect (C, H, W). The ImageNet transforms below already return that layout.


In [ ]:
class MiniImageNetDataset(Dataset):
    """Wrap a HuggingFace image dataset for use with PyTorch."""

    def __init__(self, dataset, transform):
        self.dataset = dataset
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, index):
        sample = self.dataset[index]
        image = sample["image"].convert("RGB")
        label = torch.tensor(sample["label"], dtype=torch.long)
        return self.transform(image), label


### Image transformations, datasets, and DataLoaders

Training uses random crops and horizontal flips so the model sees slightly different views of each image. Validation uses a deterministic center crop so evaluation is comparable across epochs. Both pipelines then normalize with ImageNet mean and standard deviation; those statistics keep pixel values centered near zero and match what a pretrained ImageNet model would expect.

The next two cells build `train_dataset` / `val_dataset` with those transforms, then wrap them in DataLoaders (`batch_size=32`; training shuffled, validation not).


In [ ]:
train_transform = transforms.Compose([
    transforms.Resize(256),                  # Resize the shorter side to 256
    transforms.RandomCrop(224),              # Random crop into shape 224×224
    transforms.RandomHorizontalFlip(),       # Random horizontal flip
    transforms.ToTensor(),                   # Convert to tensor, scale to [0,1], and permute into shape (C, H, W)
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]) # Normalize
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

train_dataset = MiniImageNetDataset(mini_imagenet_train, transform=train_transform)
val_dataset = MiniImageNetDataset(mini_imagenet_val, transform=val_transform)


In [ ]:
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# Inspect one training batch.
data, labels = next(iter(train_dataloader))
print(f"Shape of data: {data.shape}")       # (B, 3, 224, 224)
print(f"Shape of labels: {labels.shape}")   # (B,)


## Question 1b: Training your CNN

We are ready to train our CNN model! To train our CNN model, we need to write a training loop that passes inputs into our model, updates the weights of our model, and repeats!

**Important terminology**:
* *epoch*: one complete pass through the entire training dataset
* *batch*: a subset of samples we pass into the model at the same time
* *optimizer*: what kind of optimization algorithm we want to use (e.g. Adam, Momentum)
* *criterion*: the loss function we want to use to evaluate our model on

### Training Loops
A training loop is where your model actually learns! Each iteration through the loop, the model:
* Makes predictions
* Calculates how wrong it was (loss)
* Updates its weights to do better next time

### PyTorch Training Loops: Step-by-Step Guide and Best Practices
1. Setup - Move model to device
    1. It's a good practice to begin by moving your model to the device (either `cuda` aka GPU or `cpu`). Your model and data must be on the same device. Moving the model to the GPU can speed up training.
    
> ```python
> model.to(device)  # device = 'cuda' or 'cpu'
> ```

2. Outer Loop - Iterate for `num_epochs` epochs
    1. Models rarely learn perfectly in one pass. Multiple epochs let the model see the data repeatedly and gradually improve.

>```python
>for epoch in range(num_epochs):
>    # An epoch = one complete pass through the entire training dataset
>```

3. Set the model to training mode before passing in training data
    1. Some layers (Dropout, BatchNorm) behave differently during training vs testing.

>```python
>model.train()
>```

4. Inner loop - Iterate over training batches
    1. Process data in batches to make the most of your GPU's computational power without overflowing memory.
    2. Move the data to the same device as the model, and if needed, cast them to the appropriate dtypes.
        1. Data should have the same dtype as your model's parameters.
        2. Usually, loss functions expect the target labels to have `dtype=torch.long`.

>```python
>for x, y in train_dataloader:
>    # Process one batch at a time
>    x = x.to(device, dtype=torch.float)
>    y = y.to(device, dtype=torch.long)
>```

5. Zero out the gradients - PyTorch accumulates gradients by default. If you don't reset them, old gradients will add to new ones, causing wrong updates! **Always** zero out the gradients BEFORE the forward pass of each batch.

>```python
>optimizer.zero_grad()
>```

6. Make a forward pass to get your model's predictions on the input data.

>```python
>y_hat = model(x)
>```

7. Compute the loss between your predictions and the target labels
    1. Loss measures how wrong your predictions are. Lower loss = better predictions.
    2. The criterion will return a single number representing the loss.

>```python
>loss = criterion(y_hat, y)
>```

8. Backward pass to compute gradients
    1. Call `loss.backward()` to compute gradients for each learnable weight in your model.
9. Update the weights
    1. Call `optimizer.step()` to update the weights in your model based on the gradients computed during backpropagation!
10. Track training metrics (optional but a good practice!)
    1. Track the training loss and training accuracy
    2. Since loss is a `torch.Tensor`, we can get its scalar value by calling `loss.item()`. Otherwise, we can't directly add a `torch.Tensor` (even if it's just a one-dimensional scalar!) to another Python primitive like `train_loss`, which might be a Python float.
    3. In multi-class classification, the model's output `y_hat` is a tensor of shape `(batch_size, num_classes)` of **logits** (raw scores), not probabilities. The predicted class is `argmax` along `dim=1`. So `preds` has shape `(batch_size,)` — one predicted class for each item in the batch.
    4. To compute the training accuracy, we can check when the predicted class equals the actual true targets `y` and sum up all the correct predictions. Since `preds` and `y` are both tensors, the expression `(preds == y)` and the sum is also a `torch.Tensor`. So similarly to the loss, we can get the scalar value of this tensor by calling `.item()`.

>```python
>train_loss += loss.item()  # Accumulate loss
>preds = torch.argmax(y_hat, dim=1)  # Get predicted class for each item in the batch
>train_correct += (preds == y).sum().item()  # Count correct predictions
>```

11. Measure validation metrics after each epoch
    1. Set the model to `eval` mode - Disables dropout, uses running stats for BatchNorm, etc.
    2. Make predictions on the validation data, and measure validation loss and/or accuracy.
    3. **Important note:** Make sure to do all your validation under a `with torch.no_grad()` header! This tells PyTorch not to keep track of forward pass operations or compute gradients during this code block. This saves memory and speeds up validation because we don't want to train the model during validation–we want to measure how it's performing.

Complete training loop structure (simplified)

>```python
># === SETUP ===
>model.to(device)
>
># === EPOCH LOOP ===
>for epoch in range(num_epochs):
>    train_loss = 0.0
>    train_correct = 0.0
>    
>    # === TRAINING PHASE ===
>    model.train()  # Enable training mode
>    
>    for x, y in train_dataloader:
>        # 1. Move data to device
>        x, y = x.to(device), y.to(device)
>        
>        # 2. Reset gradients
>        optimizer.zero_grad()
>        
>        # 3. Forward pass
>        y_hat = model(x)
>        
>        # 4. Compute loss
>        loss = criterion(y_hat, y)
>        
>        # 5. Backward pass (compute gradients)
>        loss.backward()
>        
>        # 6. Update weights
>        optimizer.step()
>        
>        # 7. Track metrics (optional)
>        # ...
>    
>    # === VALIDATION PHASE ===
>    model.eval()  # Disable training-specific layers
>    with torch.no_grad():  # Don't compute gradients
>        for x, y in val_dataloader:
>            # Forward pass only, no backward pass!
>            y_hat = model(x)
>            loss = criterion(y_hat, y)
>            # Track validation metrics
>```


### How we calculate average loss and accuracy
* When we train a model, we want to report the average loss and average accuracy for each epoch, not just the total. This helps us understand how well the model is performing per sample, and lets us compare results across different epochs and experiments.

**Loss Calculation**
* During each epoch, for each batch we encounter, we add the loss computed over all items in the batch to the total loss: `train_loss += loss.item()`.
* At the end of the epoch, we divide by the number of batches: `avg_train_loss = train_loss / len(train_dataloader)`. This average is what we print and plot.
* Each batch's loss is an average over the samples in that batch, so dividing by the number of batches gives us the average loss per batch for the whole epoch.

**Accuracy Calculation**
* We count the total number of correct predictions for all samples in the epoch: `train_correct += (preds == targets).sum().item()`.
* At the end of the epoch, we divide by the total number of samples in the dataset: `train_acc = train_correct / len(train_dataloader.dataset)`.
* This gives us the fraction of samples the model predicted correctly (i.e., the overall accuracy for the epoch).

In the following block, we've implemented a training loop for you!

**Note:** `tqdm` is a Python library used to display progress bars. In the `train` function we've implemented, we use it as a wrapper around iterators such as the dataloaders so that we can track the progress of how many epochs we've trained for. It's a nice-to-have, but when you implement your own training loops in the future, you can also simply iterate through the dataloaders like in the simplified example above too.


In [ ]:
# Common practice to check if GPU is available
if IS_GRADING_ENV:
    device = "cpu"
elif torch.accelerator.is_available():
    device = torch.accelerator.current_accelerator()
else:
    device = "cpu"
print(f"Using device: {device}")

# A CPU kernel is fine for writing and shape-checking your modules, but it is far too slow
# for the training runs in this notebook, so `train` checks for an accelerator before it starts.
if not IS_GRADING_ENV and str(device) not in ["cuda", "mps"]:
    print("WARNING: no GPU/MPS accelerator found. You can keep implementing modules, but the training cells will not run.")


def train(
    model, optimizer, criterion, num_epochs, train_dataloader, val_dataloader=None
):
    """
    Args:
        model: the model to train
        optimizer: the optimizer to use
        criterion: the loss function to use
        num_epochs: the number of epochs to train for
        train_dataloader: the dataloader for the training set
        val_dataloader: the dataloader for the validation set (optional)
    """
    if IS_GRADING_ENV:
        zeros = [0.0] * num_epochs
        return zeros, zeros, zeros, zeros

    assert str(device) in ["cuda", "mps"], "Please make sure you are using an accelerator (e.g. GPU or MPS) before training"

    # === SETUP ===
    model.to(device)

    # == (Optional) Perform a validation loop before training as a baseline ==
    if val_dataloader:
        model.eval()  # Set model to evaluation mode
        num_correct = 0.0
        val_loss = 0.0
        with torch.no_grad():  # Don't compute gradients
            for x, y in val_dataloader:
                x = x.to(device, dtype=torch.float)
                y = y.to(device, dtype=torch.long)
                y_hat = model(x)

                loss = criterion(y_hat, y)  # pred, target
                val_loss += loss.item()

                preds = torch.argmax(y_hat, dim=1)
                num_correct += (preds == y).sum().item()

        avg_val_loss = val_loss / len(val_dataloader)
        val_acc = num_correct / len(val_dataloader.dataset)
        tqdm.write(f"Initial validation loss: {avg_val_loss:.4f}")
        tqdm.write(f"Initial validation accuracy: {val_acc:.4f}")

    # Lists to store metrics across epochs
    train_accuracies = []
    train_losses = []
    val_accuracies = []
    val_losses = []

    epoch_pbar = tqdm(range(num_epochs), desc="Training Progress", position=0)

    # === EPOCH LOOP ===
    for epoch in epoch_pbar:
        # === TRAINING PHASE ===
        model.train()  # Set model to training mode

        # Initialize metrics for this epoch
        train_loss = 0.0
        train_correct = 0.0

        # === INNER LOOP (iterate over training batches) ===
        for x, y in train_dataloader:
            # 1. Move data to device
            x = x.to(device, dtype=torch.float)
            y = y.to(device, dtype=torch.long)

            # 2. Reset gradients
            optimizer.zero_grad()

            # 3. Forward pass
            y_hat = model(x)

            # 4. Compute loss
            loss = criterion(y_hat, y)

            # 5. Backward pass (compute gradients)
            loss.backward()

            # 6. Update weights
            optimizer.step()

            # 7. Track metrics
            train_loss += loss.item()  # Accumulate loss (convert to Python float)
            preds = torch.argmax(
                y_hat, dim=1
            )  # Get predicted class for each item in the batch (highest prob = highest confidence)
            train_correct += (preds == y).sum().item()  # Count correct predictions

        # === END OF INNER LOOP ===

        # Compute average metrics for the epoch
        avg_train_loss = train_loss / len(
            train_dataloader
        )  # Average loss = total loss / total number of batches
        train_acc = (
            train_correct / len(train_dataloader.dataset)
        )  # Average accuracy = accuracy / total number of data points seen across the whole epoch (i.e. in the entire dataset)
        train_losses.append(avg_train_loss)
        train_accuracies.append(train_acc)

        print(f"\nEpoch {epoch + 1} - Training accuracy: {train_acc:.3f}")
        print(f"Epoch {epoch + 1} - Training loss: {avg_train_loss:.3f}")

        # Metrics to show on the progress bar (validation metrics are added below, if available)
        epoch_metrics = {
            "train_loss": f"{avg_train_loss:.3f}",
            "train_acc": f"{train_acc:.3f}",
        }

        # === END OF TRAINING PHASE ===

        # === VALIDATION PHASE ===
        if val_dataloader:
            model.eval()  # Set model to evaluation mode
            val_loss = 0.0
            val_correct = 0.0

            with torch.no_grad():  # Don't compute gradients
                # === INNER LOOP (iterate over validation batches) ===
                for x, y in val_dataloader:
                    # 1. Move data to device
                    x = x.to(device, dtype=torch.float)
                    y = y.to(device, dtype=torch.long)

                    # 2. Forward pass only, no backward pass!
                    y_hat = model(x)

                    # 3. Compute loss
                    loss = criterion(y_hat, y)

                    # 4. Track validation metrics
                    val_loss += loss.item()
                    preds = torch.argmax(y_hat, dim=1)
                    val_correct += (preds == y).sum().item()

                # === END OF INNER LOOP ===

                # Compute average metrics for the epoch
                avg_val_loss = val_loss / len(val_dataloader)
                val_acc = val_correct / len(val_dataloader.dataset)

                val_losses.append(avg_val_loss)
                val_accuracies.append(val_acc)

            # Print validation metrics for the epoch
            print(f"Epoch {epoch + 1} - Validation accuracy: {val_acc:.3f}")
            print(f"Epoch {epoch + 1} - Validation loss: {avg_val_loss:.3f}")

            epoch_metrics["val_loss"] = f"{avg_val_loss:.3f}"
            epoch_metrics["val_acc"] = f"{val_acc:.3f}"

            # === END OF VALIDATION PHASE ===

        # Show this epoch's average metrics on the progress bar
        epoch_pbar.set_postfix(epoch_metrics)

        # === END OF EPOCH LOOP ===

    return train_accuracies, train_losses, val_accuracies, val_losses


**Task:** Configure and train the CNN. Store the returned histories in `cnn_train_accuracies`, `cnn_train_losses`, `cnn_val_accuracies`, and `cnn_val_losses`; you will plot them in the next part.

1. Instantiate a CNN model which will classify which of the 10 classes from our Mini Imagenet dataset an image belongs to.
2. Use `AdamW` as the optimizer.
    1. [PyTorch documentation for AdamW](https://docs.pytorch.org/docs/stable/generated/torch.optim.AdamW.html)
3. Use `CrossEntropyLoss` as our criterion.
    1. [PyTorch documentation for CrossEntropyLoss](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html)
4. Train your CNN model for **20** epochs! Your best validation loss should reach around **1.6**.

**Hints:**
* When you instantiate an optimizer, you also need to pass in what parameters you want this optimizer to update. You can access the parameters of any class that inherits from `nn.Module` by calling its `.parameters()` function!
* Pay attention to the learning rate you pass to `AdamW`. Its default learning rate (see the documentation linked above) may be too high for this CNN. If your training accuracy is still hovering around chance (~10%) after a few epochs, try a smaller learning rate.
* Make sure that you turn on a GPU runtime before training. It may take a few minutes to complete all 20 epochs of training. For reference, the staff implementation took about 4 minutes to train on Google Colab's T4 and 2 minutes to train on Databricks' A10 for 20 epochs.

**Notes:**
* You may notice that the validation accuracy lags below the training accuracy or jumps around a lot. Since we're using a toy set with only 1,000 training images and 200 validation images, that is to be expected! These metrics are an indicator that our model might be overfitting to our training set.

In [ ]:
# TODO: Instantiate a CNN model
cnn = ...

# TODO: Set the number of epochs
num_epochs = ...

# TODO: Set the optimizer
optimizer = ...


# TODO: Set the loss criterion
criterion = ...

# TODO: Train the model. Keep these variable names; the check below reads them.
cnn_train_accuracies, cnn_train_losses, cnn_val_accuracies, cnn_val_losses = ...


In [ ]:
grader.check("q1bi")

The autograder check above only checks that `cnn_train_accuracies`, `cnn_train_losses`, `cnn_val_accuracies`, and `cnn_val_losses` are lists of length 20. To grade your model's performance, we'll plot and save the training curves. The cell below calls the provided `plot_metrics` helper on `cnn_train_losses`, `cnn_val_losses`, `cnn_train_accuracies`, and `cnn_val_accuracies`, and saves the figure for grading.


<!-- BEGIN QUESTION -->



In [ ]:
fig_q1bii = plot_metrics(
    cnn_train_losses, cnn_val_losses, cnn_train_accuracies, cnn_val_accuracies,
    num_epochs=20, title="CNN Training Metrics"
)
if not IS_GRADING_ENV:
    fig_q1bii.show()
save_figure(fig_q1bii, "q1bii")


<!-- END QUESTION -->

# Question 2: The ResNet Architecture

Deep neural networks can be incredibly powerful and expressive, but they can also be difficult to train! There are a few problems that deep neural networks encounter:
* Vanishing gradients: Earlier layers receive very small gradients because the upstream gradients get multiplied by many local gradients. Thus, during gradient descent, their weights move by very small amounts and they don't "learn" as effectively.
* Accuracy degradation: Adding more layers to a deep model causes accuracy to saturate, contributing to higher *training error* and *test error* compared to a shallower network with less layers.

Normalization can help address the vanishing gradient problem. But accuracy degradation can still persist in deep networks.

## Residual Learning

In 2015, He et al. proposed a solution to the degradation problem by introducing the concept of **residual learning**.

> [Deep Residual Learning for Image Recognition](https://arxiv.org/pdf/1512.03385) Kaiming He, Xiangyu Zhang, Shaoqing Ren, Jian Sun

Neural networks can be thought of as being composed of many different "blocks", which are just 1 or more layers (e.g. linear layers, activation functions, convolution layers, etc.) we think of as a single group.

Recall that we can view the blocks in our neural networks as functions mapping some input $x$ to some output $z$ (note that $z$ might not be the final output of our neural network like a class prediction, but could also be the input into another hidden layer).

Let's denote the ground truth function we want this block in our neural network to learn as $f$, where $f(x) = z$.

Instead of forcing the layer(s) in our block to learn $f(x)$, we let them learn another mapping $g(x)$ where $g(x) = f(x) - x$. Then, we can "add in" our original input $x$ to the output of $g(x)$ to achieve our desired mapping $f(x)$!

The authors showed that with **residual learning**:
1. Deep residual networks were easier to optimize than deep networks without residual connections
2. Residual networks demonstrate significant accuracy gains even as they get deeper, avoiding the degradation problem
3. They were able to win 1st place awards in the ImageNet detection, ImageNet localization,
COCO detection, and COCO segmentation in ILSVRC & COCO 2015 competitions.

## Benefits of Residual Learning

You might ask why we want to do this! Aren't we just trying to learn some other function $g(x)$ now? Why are we complicating things by subtracting $x$ from $f(x)$ if we have to add it back in anyways?

There are a few intuitions behind why the ResNet architecture is very powerful!

**Reason 1:** Adding a residual connection lets more of the upstream gradient flow backwards

> Imagine layer N, whose input is $x$ and whose output is $z$. Denote the upstream gradient that backpropagates from the loss $L$ to the current layer N as $\frac{\partial L}{\partial z}$. Without a residual connection, the layer computes $z = g(x)$, so the downstream gradient layer N propagates down to layer N - 1 is: $\frac{\partial L}{\partial z} \cdot \frac{\partial g}{\partial x}$.

> With a residual connection, the layer instead computes $z = g(x) + x$, so the downstream gradient that layer N propagates down to layer N - 1 is now $\frac{\partial L}{\partial z} \cdot (\frac{\partial g}{\partial x} + 1)$. Thus, even if the local gradient term $\frac{\partial g}{\partial x}$ is very small, the + 1 ensures that gradients don't vanish completely and allow information to flow backwards more effectively!


**Reason 2:** Blocks can learn residual mappings!

<div align="center">
<img style="width: 50%;" src="https://az1326.github.io/images/cs189-residual-block.jpg">
<br>
<caption>
<a href="https://d2l.ai/chapter_convolutional-modern/resnet.html">
Source: Dive Into Deep Learning
</a>
</caption>
</div>

Let's zoom in on a block of a neural network (outlined by the dotted box), with input $x$. We assume that $f(x)$, the desired ground truth mapping we want the block to learn, will be used as the input to the activation function at the top. The "weight layers" represented by the white rectangles can be fully connected linear layers or convolutional layers.

On the left, we have a block without a residual connection. The dotted-line box must directly learn $f(x)$. On the right, the layers in the dotted-line box have to learn the **residual mapping** $g(x) = f(x) - x$.

If the actual desired function we want to learn is the identity function ($f(x) = x$), then the residual mapping is just $g(x) = 0$, which is very easy for the layers on the right to learn–we just need to push the weights and biases of the "weight layer" to 0.

In conclusion, sometimes, a neural network has already learned everything it needs from its data to perform its task! Adding additional layers is like forcing the network to keep "squeezing" more information out from the data. Using a residual connection lets a network block decide whether or not it might be more effective to simply pass only $x$ forward, without applying additional non-linearities onto it.


## Question 2a: Implement a Residual Block in PyTorch

**Task:** Use PyTorch to instantiate a Residual Block with the following architecture. Do **not** hardcode 64 channels — use the `in_channels` and `out_channels` arguments passed into `__init__` (later questions reuse this block with 64, 128, 256, and 512 channels).
> * Convolution layer 1: `in_channels` → `out_channels`, 3x3 kernel, padding = 1, stride = 1 (set stride = 2 if downsampling)
> * Batchnorm (over `out_channels`)
> * ReLU activation
> * Convolution layer 2: `out_channels` → `out_channels`, 3x3 kernel, padding = 1, stride = 1
> * Batchnorm (over `out_channels`)
> * Residual / skip connection: a 1x1 convolution mapping `in_channels` → `out_channels` (stride = 2 if downsampling, else stride = 1)
> * ReLU activation

**Notes about downsampling:**
* **Downsampling** refers to reducing the size of the feature map. For instance, let's say our original input to this layer was a 28 x 28 image. We can downsample it to a 14 x 14 image, essentially halving the H and W. Downsampling is often combined with an increase in the number of output channels. The main benefits of downsampling are 1) reducing computation, 2) increasing the receptive field of the output units (i.e. how many input pixels each output unit "looks" at), and 3) enabling hierarchical feature learning (i.e. letting the model learn fine details at high resolutions, then reducing the resolution by making the input that gets passed down to later layers of a smaller size so that the model can learn bigger-picture details at a lower resolution). Many ResNet implementations downsample the input in the first convolution layer. Downsampling can be implemented by using a stride of 2 instead of a stride of 1 in a Conv2d layer. This means the kernel will shift over by 2 pixels each time instead of 1.
* When downsampling, we can't add `x` to the convolution output with no extra work because H and W won't match. **Always** define `self.residual_connection` as a 1×1 convolution (`in_channels` → `out_channels`). Use stride 2 on that 1×1 if you downsample; otherwise stride 1. Do not use `nn.Identity`: Identity cannot change the channel count, and adding tensors requires the skip path and the main path to have the same C (and the same H, W). A 1×1 conv is how the skip maps `in_channels` → `out_channels` (and, with stride 2, halves H and W). Use this 1×1 even when `in_channels == out_channels` so every block has the same skip interface.

**Hints:**
1. Use separate batchnorm layers after each convolution layer!
2. `BatchNorm2d` normalizes each channel independently over the batch and spatial dimensions, so it needs one $\gamma$ and $\beta$ per channel. When initializing a `BatchNorm2d` after a convolution, pass that convolution's **output** channel count as `num_features`.
3. The residual connection and the output of the 2 convolutions should have the same shape (i.e. the same C, H, and W dimensions). What is the shape of the output after the 2 convolutions?
    1. Apply a 1×1 convolution to the input $x$ for every ResidualBlock. A 1×1 kernel does not change H/W unless you use stride 2; it can change the channel count so C matches the main path.
4. If we want to downsample an image, we typically apply a stride of 2 in the first conv layer. If `initial_downsample = True`, make sure to adjust the stride in the relevant conv layers in the `__init__` method (the first convolution and the skip connection, not the second convolution). Also make sure to use a stride of 2 in the residual connection if we downsample.
5. Some of the functions and modules we have you implement take a `verbose` argument. It is an optional debugging flag, and the tests do not check it. To use it, store `self.verbose = verbose` in `__init__` and print tensor shapes or other debug info when `self.verbose` is true. If you call another function or module that also takes `verbose`, pass the flag through to turn on its prints too.

**Required attribute names:** `self.conv1`, `self.batchnorm1`, `self.conv2`, `self.batchnorm2`, and `self.residual_connection`. Our tests fill these layers with fixed parameters before checking `forward`. Assign those names to the `Conv2d` / `BatchNorm2d` modules themselves (not in an `nn.Sequential` wrapper).

Helpful website to calculate the output shapes after a convolution: https://madebyollin.github.io/convnet-calculator/


In [ ]:
class ResidualBlock(nn.Module):
    def __init__(
        self,
        in_channels=3,
        out_channels=64,
        kernel_size=3,
        initial_downsample=False, # whether to downsample the input
        verbose=False, # whether to print debug statements
    ):
        super().__init__()
        self.verbose = verbose


        # TODO: Define the first conv layer
        self.conv1 = ...
        # TODO: Define the first batchnorm layer
        self.batchnorm1 = ...
        # TODO: Define the second conv layer
        self.conv2 = ...
        # TODO: Define the second batchnorm layer
        self.batchnorm2 = ...
        # TODO: Define the residual connection
        self.residual_connection = ...

    def forward(self, x):
        # TODO: Implement the forward pass
        conv1 = ...
        if self.verbose:
            print(f"conv1 shape: {conv1.shape}")
        bn1 = ...
        if self.verbose:
            print(f"bn1 shape: {bn1.shape}")
        z1 = ...

        conv2 = ...
        if self.verbose:
            print(f"conv2 shape: {conv2.shape}")
        bn2 = ...
        if self.verbose:
            print(f"bn2 shape: {bn2.shape}")

        residual = ...
        if self.verbose:
            print(f"residual shape: {residual.shape}")

        out = ...
        # Don't forget to apply ReLU activation after adding the residual
        out = ...
        return out


In [ ]:
grader.check("q2a")

### Check: Residual Block output shape

Run this cell **after** you have implemented `ResidualBlock.forward`.

Below, we create a tensor of shape `(1, 3, 224, 224)` (batch, channels, height, width) filled with random numbers between [0, 1] and pass it through our `ResidualBlock`. Let's do some math to determine what the expected dimensions of our `ResidualBlock` output should be!

Let's say we instantiate a `ResidualBlock` with 3 input channels, 64 output channels per hidden convolution, a kernel size of 3, and initial downsampling during the first convolution. These would be the shapes after each layer:
1. Input: 1 x 3 x 224 x 224 --> dimensions: Batch size x 3 RGB channels x height of 224 pixels x width of 224 pixels
2. After conv1: 1 x 64 x 112 x 112 --> the first convolution downsamples the image to half its original height and width, and also increases the number of channels to 64
3. After bn1: 1 x 64 x 112 x 112 --> batchnorm doesn't change the output dimensions
4. After the first ReLU: 1 x 64 x 112 x 112 --> ReLU doesn't change the output dimensions
5. After conv2: 1 x 64 x 112 x 112 --> the second conv has padding = 1. This is called *same padding* because we pad the input so that the output of this layer has the same dimensions.
6. After batch norm 2: 1 x 64 x 112 x 112 --> batchnorm doesn't change the output dimensions
7. After adding the residual: 1 x 64 x 112 x 112
8. After the final ReLU: 1 x 64 x 112 x 112 --> ReLU doesn't change the output dimensions

You should verify that the shape of your actual output is also 1 x 64 x 112 x 112!


In [ ]:
x = torch.rand((1, 3, 224, 224))  # 1 image, 3 RGB channels, 224 x 224 pixels
print(f"Shape of input: {x.shape}")

res = ResidualBlock(in_channels=3, out_channels=64, kernel_size=3, initial_downsample=True, verbose=False)
resblock_out = res(x)
print(f"Shape of output: {resblock_out.shape}")
assert resblock_out.shape == torch.Size([1, 64, 112, 112]), f"Expected output shape (1, 64, 112, 112), but got {resblock_out.shape}"


In [ ]:
# Verify the shapes of the weights
conv1_shape = res.conv1.weight.shape
print(f"Shape of conv1 weight: {conv1_shape}")
assert conv1_shape == torch.Size([64, 3, 3, 3]), f"Expected Conv1 weights to have shape (64, 3, 3, 3), but got {conv1_shape}"

conv2_shape = res.conv2.weight.shape
print(f"Shape of conv2 weight: {conv2_shape}")
assert conv2_shape == torch.Size([64, 64, 3, 3]), f"Expected Conv2 weights to have shape (64, 64, 3, 3), but got {conv2_shape}"

residual_shape = res.residual_connection.weight.shape
print(f"Shape of residual weight: {residual_shape}")
assert residual_shape == torch.Size([64, 3, 1, 1]), f"Expected residual connection to have shape (64, 3, 1, 1), but got {residual_shape}"


## Question 2b: ResNet-18(9)

Awesome! We've put together one building block of a ResNet. Now, let's implement a full ResNet-18 architecture using PyTorch.

ResNet-18 is an implementation of ResNet that has 18 total learnable layers: an initial 7x7 convolution, a series of Residual Blocks with 2 convolutions each, and a final fully connected layer for classification. There also exist other models in the ResNet family such as ResNet34, ResNet 50, and even ResNet 152!

### ResNet 18 Architecture

ResNet-18 has 4 main stages, each with a different number of residual blocks. You should use the `ResidualBlock` class you implemented above to build the ResNet-18 architecture.

---

#### 1. Initial Convolution Layer
- **Convolution Layer:** 3 input channels, 64 output channels, kernel size 7×7, stride 2, padding 3
- **BatchNorm:** Normalize 64 channels
- **ReLU Activation:** Apply ReLU after BatchNorm
- **MaxPooling:** 3×3 kernel, stride 2, padding 1

#### 2. Stage 1: 2 Residual Blocks (64 channels)
- **Residual Block 1:**
  - Input: 64 channels → Output: 64 channels
  - Kernel size: 3×3, stride 1
  - Uses the 1×1 skip already inside `ResidualBlock` (always present; not an extra layer you add here)
- **Residual Block 2:**
  - Input: 64 channels → Output: 64 channels  
  - Kernel size: 3×3, stride 1
  - Uses the 1×1 skip already inside `ResidualBlock`

#### 3. Stage 2: 2 Residual Blocks (128 channels)
- **Residual Block 3:**
  - Input: 64 channels → Output: 128 channels
  - Kernel size: 3×3, stride 2 (downsampling)
  - Uses the 1×1 skip already inside `ResidualBlock`
- **Residual Block 4:**
  - Input: 128 channels → Output: 128 channels
  - Kernel size: 3×3, stride 1
  - Uses the 1×1 skip already inside `ResidualBlock`

#### 4. Stage 3: 2 Residual Blocks (256 channels)
- **Residual Block 5:**
  - Input: 128 channels → Output: 256 channels
  - Kernel size: 3×3, stride 2 (downsampling)
  - Uses the 1×1 skip already inside `ResidualBlock`
- **Residual Block 6:**
  - Input: 256 channels → Output: 256 channels
  - Kernel size: 3×3, stride 1
  - Uses the 1×1 skip already inside `ResidualBlock`

#### 5. Stage 4: 2 Residual Blocks (512 channels)
- **Residual Block 7:**
  - Input: 256 channels → Output: 512 channels
  - Kernel size: 3×3, stride 2 (downsampling)
  - Uses the 1×1 skip already inside `ResidualBlock`
- **Residual Block 8:**
  - Input: 512 channels → Output: 512 channels
  - Kernel size: 3×3, stride 1
  - Uses the 1×1 skip already inside `ResidualBlock`

#### 6. Classification Head
- **Global Average Pooling:** `nn.AdaptiveAvgPool2d((1, 1))`
- **Flatten Layer:** Convert the output of the global average pooling to a 1D value for each channel
- **Fully Connected Layer:** 512 input features → num_classes output

---

#### Key Implementation Notes
- **Downsampling:** The first residual block in stages 2, 3, and 4 use stride=2 to downsample the input while simultaneously increasing the number of channels. Combining downsampling with an increase in the channel count allows the network to extract more high-level features and reduces computational costs because smaller feature maps are faster to process.
- **Channel Changes:** The first residual block in stages 2-4 double the channel count of the input
- **Skip Connections**: Do not add extra skip layers in `ResNet18`. Every `ResidualBlock` already has `self.residual_connection` as a 1×1 conv, including when in/out channels match.

---

**Task:** Fill in the `__init__` and `forward` method of the ResNet18 class.

**Required structure:** Use `self.initial_conv` for the initial convolution layer, `self.residual1` through `self.residual4` for the four residual stages, and `self.classification_head` for the classification head. These correspond to the six parts described above. Each of these should be a  [`nn.Sequential`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Sequential.html), which applies the layers you pass it in order.

**Hints:**
1. Note that the stages are composed of multiple residual blocks in sequence. In the ResNet-18 implementation, whenever we want to downsample or change the number of channels, we do that in the first residual block of each stage.

2. You can use AdaptiveAvgPool2d to get a 1x1 feature map for the final fully-connected layer without having to specify the input_size, kernel_size, stride, padding, etc. explicitly.

3. You can use nn.Flatten() to flatten the output of the global average pooling layer to a 1D value for each channel.


In [ ]:
class ResNet18(nn.Module):
    def __init__(self, num_classes, verbose=False):
        super().__init__()

        # TODO: Instantiate all the layers/stages of ResNet-18
        # Tests require these names. Each one is an nn.Sequential.
        self.initial_conv = ...
        self.residual1 = ...
        self.residual2 = ...
        self.residual3 = ...
        self.residual4 = ...
        self.classification_head = ...

    def forward(self, x):

        # TODO: Implement the forward pass of ResNet-18
        ...
        return ...


In [ ]:
grader.check("q2b")

### Check: ResNet output shape

Run this cell **after** you have implemented `ResNet18.forward`.

Once again, let's check the output shape of our ResNet.

What should be the output's expected shape?

If the dimensions and channels of all our intermediate convolution layers match up, we should get an output with shape (Batch_size, 10) since we specified 10 as the number of classes!


In [ ]:
x = torch.rand((1, 3, 224, 224))  # 1 image, 3 RGB channels, 224 x 224 pixels
print(f"Shape of input: {x.shape}")

resnet18 = ResNet18(num_classes=10, verbose=True)
resnet18_out = resnet18(x)
print(f"Shape of output: {resnet18_out.shape}")
assert resnet18_out.shape == torch.Size([1, 10]), f"Expected output shape (1, 10), but got {resnet18_out.shape}"


## Question 2c: Training ResNet-18

**Task:** Configure and train ResNet-18. Store the returned histories in `resnet_train_accuracies`, `resnet_train_losses`, `resnet_val_accuracies`, and `resnet_val_losses`; you will plot them in the next part.

1. Instantiate a ResNet18 model which will classify which of the 10 classes from our Mini Imagenet dataset an image belongs to.
2. Use `AdamW` as the optimizer.
    1. [PyTorch documentation for AdamW](https://docs.pytorch.org/docs/stable/generated/torch.optim.AdamW.html)
3. Use `CrossEntropyLoss` as our criterion.
    1. [PyTorch documentation for CrossEntropyLoss](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html)
4. Train your ResNet18 model for **50** epochs! Your best validation loss should reach around **1.4**.

**Hints:**
* When you instantiate an optimizer, you also need to pass in what parameters you want this optimizer to update. You can access the parameters of any class that inherits from `nn.Module` by calling its `.parameters()` function!
* Make sure that you turn on a GPU runtime before training. It may take a few minutes to complete all 50 epochs of training. For reference, the staff implementation took about 15 minutes on Google Colab's T4 and 7 minutes on Databricks' A10 to train for 50 epochs.

**Notes:**
* You may notice that the validation accuracy lags below the training accuracy or jumps around a lot. Since we're using a toy set with only 1,000 training images and 200 validation images, that is to be expected! These metrics are an indicator that our model might be overfitting to our training set.

In [ ]:
# TODO: Instantiate a ResNet18 model
resnet = ...

# TODO: Set the number of epochs
num_epochs = ...

# TODO: Instantiate an optimizer
optimizer = ...


# TODO: Instantiate a loss criterion
criterion = ...

# TODO: Train the model. Keep these variable names; the check below reads them.
resnet_train_accuracies, resnet_train_losses, resnet_val_accuracies, resnet_val_losses = ...


In [ ]:
grader.check("q2ci")

Now we'll plot and save our training curves. The cell below calls the provided `plot_metrics` helper on `resnet_train_losses`, `resnet_val_losses`, `resnet_train_accuracies`, and `resnet_val_accuracies`, and saves the figure for grading.


<!-- BEGIN QUESTION -->



In [ ]:
fig_q2cii = plot_metrics(
    resnet_train_losses, resnet_val_losses,
    resnet_train_accuracies, resnet_val_accuracies,
    num_epochs=50, title="ResNet Training Metrics"
)
if not IS_GRADING_ENV:
    fig_q2cii.show()
save_figure(fig_q2cii, "q2cii")


<!-- END QUESTION -->

# Question 3: Transformers

<div align="center">
<img style="width: 50%;" src="https://machinelearningmastery.com/wp-content/uploads/2021/08/attention_research_1.png" />
<br>
The original encoder-decoder Transformer (source: Machine Learning Mastery)
</div>

Like ResNets, Transformers are another architecture innovation that *transformed* the world of machine learning.

Transformers were first invented for sequence-to-sequence tasks like machine translation: take one sequence (e.g. an English sentence) and output another (e.g. its French translation). Earlier models such as [RNNs (recurrent neural networks)](https://d2l.ai/chapter_recurrent-neural-networks/index.html) read the input one token at a time and passed information along in a hidden vector, so they struggled with long-range context and were slow to train. Transformers replace that step-by-step processing with *attention*, which lets every position look directly at the parts of the sequence that matter most for predicting what comes next.

## Attention

Recall that the dot product of two vectors measures how similar they are. Transformers build on this idea with *scaled dot-product attention*:

$$Attention(Q, K, V) = softmax(\frac{Q K^T}{\sqrt{d_k}}) V$$

* Each token gets a query vector $q$, a key vector $k$, and a value vector $v$, which are learned linear projections of its embedding. Stacking them row by row gives the matrices $Q$, $K$, and $V$.
    * A helpful analogy is a library! The *query* is what you're searching for (say, a book about matrix calculus), the *keys* are the titles you compare your search against, and the *values* are the actual contents of the books.
* The $(i, j)$-th entry of $Q K^T$ scores how relevant the $j$-th token is to the $i$-th token. We divide by $\sqrt{d_k}$ to keep the variance of these scores from growing with the key dimension.
* The softmax turns each row of scores into attention weights that sum to 1.
* Multiplying by $V$ gives each token a weighted sum of the value vectors: a new representation built from the tokens it attends to.

## Encoders and Decoders

The original Transformer from *Attention Is All You Need* has two halves:
* The **encoder** reads the full input sequence. Its self-attention lets every token attend to every other token, both before and after it.
* The **decoder** generates the output one token at a time. Its self-attention is *masked* so each position only sees earlier positions, and a *cross-attention* sublayer lets it read the encoder's output.

Each layer in both halves also has a position-wise feed-forward network, with residual connections and LayerNorm around every sublayer.

Most models today keep only one half:
* **Encoder-only** models, like BERT, produce contextual embeddings of an entire input for tasks like classification. You'll fine-tune one (DNABERT) in Homework 4.2.
* **Decoder-only** models, like GPT, predict the next token from the tokens before it, which is all you need to generate text.

In this problem, you will implement the most common form of transformer today: a **decoder-only** transformer. You'll build it up from softmax and attention, then train it to write short stories.

## Question 3a: Softmax

The softmax function converts a vector of real numbers between $[-\infty, \infty]$ into probabilities that sum up to one.

For a vector **$z$** = $[z_1, z_2, \ldots, z_k]$, the softmax output for the $i$-th element in $z$ is given by:

$$softmax(z_i) = \frac{e^{z_i}}{\sum_{j=1}^{k} e^{z_j}}$$

Each component $softmax(z_i) \in (0, 1)$ and all the components sum to 1.

Sometimes, you will also see the softmax output written as
$$softmax(z_i) = \frac{e^{z_i - max(z)}}{\sum_{j=1}^{k} e^{z_j - max(z)}}$$

where the maximum value of **$z$** is subtracted from each element before exponentiation. By subtracting max(z) from each input, exponentials are all kept between 0 and 1 so large numbers won't overflow into infinity and small numbers won't underflow to 0. This is sometimes used as a more numerically stable version of the softmax function for floating point arithmetic.

**Task:** Implement the softmax function (either the vanilla version, or stabilized version)

**Hints:**
* The input $x$ might be a 1-D tensor of shape `(d,)` or a higher-rank tensor (for example `(B, N, d)`). Softmax over the **last** dimension: use `dim=-1` and `keepdim=True` when summing so the result broadcasts.
* Some helpful PyTorch functions:
    * `torch.exp` [documentation](https://docs.pytorch.org/docs/stable/generated/torch.exp.html)
    * `torch.sum` [documentation](https://docs.pytorch.org/docs/stable/generated/torch.sum.html)


In [ ]:
def softmax(x: torch.Tensor):
    """
    Compute the softmax of each element in x.
    """
    ...


In [ ]:
grader.check("q3a")

### Check
Run this cell **after** you have implemented `softmax`. To check the correctness of our implementation, we can:
* Verify that the output of our softmax function has the expected dimensions (the softmax output should have the same shape as the input)
* Verify that the elements along the last dimension sum up to 1 (since softmax should transform them into a valid probability distribution)
* Compare our implementation with PyTorch's `softmax` function and check that our values are within a reasonable range of PyTorch's


In [ ]:
set_seed(SEED)

rand_1d = torch.randn(5)
print(f"Random 1D vector with shape (d): {rand_1d.shape}")
out_1d = softmax(rand_1d)

print(f"Softmax output shape: {out_1d.shape}\nExpected shape: {rand_1d.shape}\n")

# Assert that the sum of the softmax output along the last dimension is 1
assert torch.allclose(torch.sum(out_1d, dim=-1), torch.ones_like(out_1d[..., 0]), rtol=1e-5), "Sum of softmax output along last dimension is not 1"

rand_3d = torch.randn(2, 5, 5)
print(f"Random vector with shape (B, N, N): {rand_3d.shape}")
out_3d = softmax(rand_3d)
print(f"Softmax output shape: {out_3d.shape}\nExpected shape: {rand_3d.shape}\n")

# Assert that the sum of the softmax output along the last dimension is 1
assert torch.allclose(torch.sum(out_3d, dim=-1), torch.ones_like(out_3d[..., 0]), rtol=1e-5), "Sum of softmax output along last dimension is not 1"

# Compare to torch implementation
torch_softmax_1d = torch.nn.functional.softmax(rand_1d, dim=-1)
assert torch.allclose(torch_softmax_1d, out_1d, rtol=1e-5), "Softmax implementation for 1D tensor does not match torch implementation"

torch_softmax_3d = torch.nn.functional.softmax(rand_3d, dim=-1)
assert torch.allclose(torch_softmax_3d, out_3d, rtol=1e-5), "Softmax implementation for 3D tensor does not match torch implementation"


## Question 3b: Scaled Dot Product Attention

$$Attention(Q, K, V) = softmax(\frac{Q K^T}{\sqrt{d_k}}) V$$

Before we dive into implementing the scaled dot product attention mechanism, let's review what **masking** is! Masking in attention serves 2 critical purposes:
1. Causal Masking (Look-Ahead Mask) prevents future information leakage. In decoder-only models (like GPT) and the decoder portion of encoder-decoder models, we need to ensure that when we are at position $i$ predicting the next token, the model can only attend to tokens at positions $\leq i$ and not future positions.
    1. Why? - During training, we have access to the entire sequence, but at inference time, we generate tokens one at a time and don't know future tokens yet
    2. Without masking, the model would "cheat" by looking at future tokens during training. This creates a mismatch between training and inference behavior

>Sentence = "The cat sat on"
>When at "sat" (looking to predict the next token "on"), the model should only see:
>  ✓ "The"
>  ✓ "cat"
>  ✓ "sat"
>  ✗ "on"  (future token - must be masked)

2. Padding Mask - When batching sequences of different lengths, we sometimes pad shorter sequences with special tokens. We need to mask these padding positions so the model doesn't attend to meaningless values.
    1. Why? - Padding tokens have no semantic meaning, so they shouldn't influence what the model attends to. We mask [PAD] tokens so they receive zero attention weight.

> Batch of 2 sequences:
> Seq 1: "Hello world"          [length 2]
> Seq 2: "I love dinosaurs"     [length 3]
>
>After padding:
>Seq 1: "Hello world [PAD]"
>Seq 2: "I love dinosaurs"

We implement masking by setting the attention score for positions we want to mask out to $-inf$ **before** we apply softmax. After the softmax operation, these values become 0.

**Task:**
1. Implement the `scaled_dot_product_attention` function which computes the scaled dot product attention of $Q$, $K$, and $V$.
2. If `mask` is not None, use `masked_fill` to fill all values where `mask = True` with `float('-inf')`.
3. Use the `softmax` function you implemented above to compute the softmax.

**Hints:**
* Make sure to properly transpose `K` (which axes need to be swapped?) before matrix-multiplying with `Q`. See the documentation for [`torch.transpose`](https://docs.pytorch.org/docs/stable/generated/torch.transpose.html).
* Use [`masked_fill`](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.masked_fill.html) (out-of-place) to fill masked values with `float('-inf')`. Prefer `masked_fill` over in-place `masked_fill_`.
* Don't forget to divide $Q K^T$ by the square root of the query/key vectors' dimension ($d_k$) before applying `softmax`.
* `d_k` comes from `Q.shape[-1]`, so it is a Python int. `torch.sqrt` expects a tensor, and `torch.sqrt(d_k)` raises `TypeError`. An integer tensor does work: `torch.sqrt(torch.tensor(d_k))` returns a float tensor. Dividing by the Python float `d_k ** 0.5` works too.


In [ ]:
def scaled_dot_product_attention(Q: torch.Tensor, K: torch.Tensor, V: torch.Tensor, mask=None, verbose=False):
    """
    Q: matrix of shape (B, N_q, d_k)
    K: matrix of shape (B, N_k, d_k)
    V: matrix of shape (B, N_k, d_v)
    mask: boolean tensor broadcastable to (B, N_q, N_k). Values where mask is True will be masked out

    B is the batch size
    N_q is the query sequence length; N_k is the key/value sequence length (equal for self-attention; may differ for cross-attention)
    d_k is the dimensions of the query/key vectors
    d_v is the dimensions of the value vectors
    """
    assert Q.shape[-1] == K.shape[-1], "Q and K must have same d_k dimension"
    assert K.shape[-2] == V.shape[-2], "K and V must have same sequence length"


    # TODO: Implement the attention function
    ...


In [ ]:
grader.check("q3b")

### Check

Run this cell **after** you have implemented `scaled_dot_product_attention`. Once again, let's perform a quick sanity check by checking the shape of the scaled dot product attention's output and comparing it to PyTorch's implementation.


In [ ]:
Q_tensor = torch.randn(2, 5, 10)
K_tensor = torch.randn(2, 5, 10)
V_tensor = torch.randn(2, 5, 5)

print(f"Shape of Q: {Q_tensor.shape}")
print(f"Shape of K: {K_tensor.shape}")
print(f"Shape of V: {V_tensor.shape}")

sdpa_output = scaled_dot_product_attention(Q_tensor, K_tensor, V_tensor)
print(f"Shape of output: {sdpa_output.shape}")
assert sdpa_output.shape == torch.Size([2, 5, 5]), f"Expected output shape (B, N, d_v) {Q_tensor.shape}, but got {sdpa_output.shape}"

# Test that implementation is equal to torch implementation
torch_sdp = torch.nn.functional.scaled_dot_product_attention(
    Q_tensor, K_tensor, V_tensor, attn_mask=None
)
assert torch.allclose(sdpa_output, torch_sdp, rtol=1e-5), "Scaled dot product attention implementation does not match torch implementation"


## Question 3c: Attention Head

Similarly to how we implemented our own ResidualBlocks that inherited from the `nn.Module` class, we can do the same for each part of a Transformer!

We'll start by implementing a single attention head.

**Task:**
1. Implement the `__init__` method:
    1. Initialize the linear transformation matrices for queries, keys, and values.
2. Implement the `forward` method:
    1. Project `x` into `Q`, `K`, and `V`. In self-attention, all three come from the same input `x`.
    2. Compute attention with the `scaled_dot_product_attention` function you defined above, passing along `mask` if provided.

**Hints:**
* Use `nn.Linear` layers named **`self.W_q`**, **`self.W_k`**, and **`self.W_v`** in `__init__`. Tests load fixed weights into these attributes. These should be the `Linear` modules themselves, not wrapped in `nn.Sequential`.

**Expected behavior:**
* Input shape: (batch_size, seq_len, d_model)
* Output shape: (batch_size, seq_len, d_k) where d_k is the dimension of the key/value vectors

In [ ]:
class AttentionHead(nn.Module):
    def __init__(self, d_model=512, d_k=64, verbose=False):

        # TODO: Implement the __init__ method
        # Don't forget to call super().__init__()!
        # Tests require these names:
        self.W_q = ...
        self.W_k = ...
        self.W_v = ...

    def forward(self, x, mask=None):
        """
        x is used for the queries, keys, and values (self-attention).
        mask is the mask to use for the attention
        """


        # TODO: Implement the forward method
        ...

In [ ]:
grader.check("q3c")

In [ ]:
# Sanity check the shape of the Attention Head output (run after implementing AttentionHead.forward)
set_seed(SEED)
batch_size = 10
seq_len = 20
d_model = 512
d_k = 64

x = torch.rand((batch_size, seq_len, d_model))
print(f"Shape of x: {x.shape}")

ah = AttentionHead(d_model=d_model, d_k=d_k)

ah_out = ah(x)

print(f"Attention head output shape: {ah_out.shape}")

assert ah_out.shape == torch.Size([batch_size, seq_len, d_k]), f"Expected output shape (B, N, d_k) {batch_size, seq_len, d_k}, but got {ah_out.shape}"


## Question 3d: Multi-Head Attention

### What is Multi-Headed Attention?

Multi-headed attention runs several attention “heads” in parallel. Each head:
* Projects inputs into its own Q, K, V subspace of size `d_k = d_model / num_heads`
* Computes scaled dot-product attention independently
* Outputs a sequence of shape `(batch, seq_len, d_k)`
These head outputs are concatenated back to `(batch, seq_len, d_model)` and passed through a final linear layer to mix information across heads.

Expected shapes:
* Input: `(B, seq_len, d_model)`
* Per head: `(B, seq_len, d_k)`
* Concat: `(B, seq_len, h·d_k)` = `(B, seq_len, d_model)`
* Output projection: `(B, seq_len, d_model)`

Benefits of Multi-Headed Attention
* Multiple perspectives: Different heads can learn different relationships simultaneously (e.g., syntax vs. semantics, short-range vs. long-range).
* Subspace specialization: Learned projections let heads focus on complementary features in different representation subspaces.
* Expressivity without huge cost: Multiple small heads approximate richer interactions than a single large head, with similar compute.
* More stable learning: Lower-dimensional per-head dot products and diversity across heads can stabilize training and reduce overfitting to one pattern.

In practice, multi-head attention helps models capture richer, multi-scale dependencies across tokens, improving performance on a wide range of sequence tasks.

---

**Task:** Implement the `MultiHeadAttention` module
1. Implement the `__init__` method.
    1. Compute the dimensions of the query and key vectors in each head (`d_k`).
    2. Instantiate `num_heads` `AttentionHead` modules with `d_k`, stored as **`self.attention_heads`** (`nn.ModuleList`).
    3. Instantiate a linear projection **`self.W_out`** from `d_model` to `d_model`.
2. Implement the `forward` method.
    1. Pass the same `x` and `mask` (if provided) through each head.
    2. Concatenate the outputs of all the heads along the last dimension.
    3. Pass the concatenated output into `W_out` to compute the final linear projection.

**Hints:**
* `d_model` refers to the embedding/hidden size of the model’s representations. You can calculate the per-head dimension `d_k` for each head's queries/keys by dividing `d_model` by `num_heads`.
* If you are storing all your `AttentionHeads` in a plain Python list, PyTorch won't know to register their parameters later on during training! Instead, you should store them in `nn.ModuleList`.
* Each head output has shape `(batch_size, seq_len, d_k)`. What dimension should you concatenate them in to get an output with shape `(batch_size, seq_len, d_model)`?

**Expected behavior:**
* Input: `x` shape `(batch_size, seq_len, d_model)`
* Output: shape `(batch_size, seq_len, d_model)`

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(
        self,
        num_heads=8,
        d_model=512,  # dimension of the embeddings
        verbose=False,
    ):
        """
        Args:
            num_heads: number of attention heads
            d_model: dimension of the embeddings
            verbose: whether to print debug information
        """

        # TODO: Implement the __init__ method
        # Don't forget to call super().__init__()!
        # Tests require these names:
        self.attention_heads = ...
        self.W_out = ...

    def forward(self, x, mask=None):

        # TODO: Implement the forward method
        ...

In [ ]:
grader.check("q3d")

In [ ]:
# Sanity check the shape of the Multi-Head Attention output (run after implementing MultiHeadAttention.forward)
set_seed(SEED)
batch_size = 10
seq_len = 20
d_model = 512
num_heads = 8

x = torch.rand((batch_size, seq_len, d_model))
print(f"Shape of x: {x.shape}")

mhsa = MultiHeadAttention(num_heads=num_heads, d_model=d_model, verbose=False)

mhsa_out = mhsa(x)

print(f"Shape of Multi-Head Attention output: {mhsa_out.shape}")

assert mhsa_out.shape == torch.Size([batch_size, seq_len, d_model]), f"Expected output shape (B, N, d_model) {batch_size, seq_len, d_model}, but got {mhsa_out.shape}"


## Question 3e: Decoder Layer

A decoder layer combines masked self-attention, a feedforward network, residual connections, and layer normalization. A decoder-only transformer is just a stack of these layers.

### Feedforward Network (FFN)
The FFN is a small multi-layer perceptron applied independently at each sequence position (same weights at every position). It typically follows this structure: `Linear(d_model → d_ffn) → nonlinearity (ReLU/GELU) → Linear(d_ffn → d_model)`.

`d_ffn` is the hidden/inner width of the FFN (the expansion dimension). Generally, `d_ffn` is chosen to be `4 * d_model`. After the first linear layer "expands" each vector from dimension `d_model` to `d_ffn`, the second linear layer "shrinks" the expanded vector back down to `d_model` so that it can be passed to the next layer.

You might ask, why would we expand the dimensions up to `d_ffn` if we're just going to shrink it back down to `d_model` again? The reason is that generally, vectors in higher-dimensions can encode more diverse types of information. We're sort of giving the FFN a chance to learn additional information about the vectors in this new, bigger feature space that it can then compress into a more useful vector even back in a feature space with only `d_model` dimensions.

### Pre-norm
*Attention Is All You Need* uses **post-norm**: each sublayer's output is added to its input, and then the sum is normalized.

$$x \leftarrow \text{LayerNorm}(x + \text{Sublayer}(x))$$

Almost every transformer trained today uses **pre-norm** instead: normalize the input first, run the sublayer, and add the result back to the unnormalized input.

$$x \leftarrow x + \text{Sublayer}(\text{LayerNorm}(x))$$

With pre-norm, the residual path from a layer's input to its output never passes through a LayerNorm, so gradients flow straight through deep stacks of layers. This makes pre-norm models easier to train and less sensitive to the learning rate. Because nothing normalizes that residual path, pre-norm models apply one final LayerNorm after the last layer. You'll add it in Question 3g.

### Decoder Layer Architecture
1. LayerNorm 1
2. Masked MultiHeadAttention with `d_model` model dimensions and `num_heads` heads. During `forward`, we use a look-ahead mask so each position can only attend to itself and earlier positions.
3. A residual connection to add the layer's input `x` to the masked self-attention's output
4. LayerNorm 2
5. A position-wise feedforward network (FFN) which consists of:
    1. A linear layer with `d_model` in_features and `d_ffn` out_features
    2. A ReLU activation
    3. A linear layer with `d_ffn` in_features and `d_model` out_features
6. A residual connection to add the output of step 3 to the FFN's output

**Task:** Implement the DecoderLayer module.
1. Implement the `__init__` method.
    1. Instantiate **`self.layernorm1`**, a `LayerNorm` over the last dimension `d_model`, and **`self.masked_self_attention`**, a `MultiHeadAttention` with `d_model` and `num_heads`.
    2. Instantiate **`self.layernorm2`**, a `LayerNorm` over `d_model`, and **`self.ffn`**, a 2-layer FFN with hidden size `d_ffn = 4 * d_model` and a ReLU between the two linear layers.
2. Implement the `forward` method.
    1. Build a causal/look-ahead mask of shape `(seq_len, seq_len)` that is `True` on **future** positions. Use `torch.triu(..., diagonal=1)` so the current token can still attend to itself (`diagonal=0` would also mask the current token).
    2. Normalize `x` with `layernorm1`, pass the result and the mask through masked self-attention, and add the attention output to the original `x`.
    3. Normalize that sum with `layernorm2`, pass the result through the FFN, and add the FFN output to the sum from the previous step.
    4. Return the result. There is no LayerNorm at the end of the layer.

**Hints:**
* `torch.triu` ([documentation](https://docs.pytorch.org/docs/stable/generated/torch.triu.html#torch.triu)) with **`diagonal=1`** builds the look-ahead mask (True = masked out).
* Make sure any tensor you create inside `forward` (such as the look-ahead mask) is on the same device as the input tensor `x`.
* You can group all the layers of the FFN together using `nn.Sequential`: [documentation](https://docs.pytorch.org/docs/stable/generated/torch.nn.Sequential.html).
* When instantiating a `LayerNorm`, you need to pass in a parameter called `normalized_shape`. `normalized_shape` specifies the size of the last dimension over which LayerNorm will compute mean/variance and learn a scale/bias parameter. What is the last dimension of each LayerNorm's input?
* Each residual connection adds the sublayer's output to the **unnormalized** tensor that went into the LayerNorm.

**Expected behavior:**
* Input: `x` shape `(batch_size, seq_len, d_model)`
* Output: shape `(batch_size, seq_len, d_model)`

In [ ]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model=512, num_heads=8, verbose=False):

        # TODO: Implement the __init__ method
        # Don't forget to call super().__init__()!
        # Tests require these names:
        self.layernorm1 = ...
        self.masked_self_attention = ...
        self.layernorm2 = ...
        self.ffn = ...

    def forward(self, x):

        # TODO: Implement the forward method

        # 1. Create a look-ahead mask

        # 2. LayerNorm, masked self-attention, residual connection

        # 3. LayerNorm, feedforward network, residual connection

        # 4. Return the output
        return ...

In [ ]:
grader.check("q3e")

In [ ]:
# Sanity check the shape of the DecoderLayer output (run after implementing DecoderLayer.forward)
set_seed(SEED)
batch_size = 10
seq_len = 20
d_model = 512
num_heads = 8

x = torch.rand((batch_size, seq_len, d_model)) # (B, N, d_model)
print(f"Shape of x: {x.shape}")

decoder = DecoderLayer(d_model=d_model, num_heads=num_heads, verbose=True)
decoder_output = decoder(x)

print(f"Shape of decoder layer output: {decoder_output.shape}")

assert decoder_output.shape == torch.Size([batch_size, seq_len, d_model]), f"Expected output shape (B, N, d_model) {batch_size, seq_len, d_model}, but got {decoder_output.shape}"

## Question 3f: Positional Encodings

### What are Positional Encodings?

The last building block of a transformer we'll need is a positional encoding layer. Previous seq2seq architectures processed input tokens one-by-one. However, every word/token in the  source input sequence is passed to a transformer at once. So how does the transformer know the order of the words in the sequence?

Positional encodings add token-position information into the embeddings of the tokens. This helps the model know the ordering of the tokens within the source sequence.

In the original Transformer paper, positional encodings are implemented using $sine$ and $cosine$ waves with different frequencies depending on the position of the token and which element within that token's embedding we are looking at.

$$PE_{pos, 2i} = sin(\frac{pos}{10000^{\frac{2i}{d_{model}}}})$$
$$PE_{pos, 2i + 1} = cos(\frac{pos}{10000^{\frac{2i}{d_{model}}}})$$

In this equation, $pos$ refers to the position of a given token in the sequence and $i$ refers to which dimension/index of the token's embedding this value applies to.

Many models today learn their positional embeddings or use rotary position embeddings (RoPE) instead. We'll use the sinusoidal version because it needs no training and is easy to inspect.

**Task:** Implement the `PositionalEncoding` module.
1. Implement `__init__(d_model, max_len)`.
    1. Instantiate **`self.pos`**, a `(max_len, d_model)` tensor of dummy values.
    2. Fill even embedding indices with `sin(pos / 10000^(2i/d_model))` and odd embedding indices with `cos(pos / 10000^(2i/d_model))`. The exponent `2i / d_model` is the same for both; only the trig function differs.
2. Implement `forward(x)`.
    1. Add `self.pos[:seq_len]` to `x` (slice to the current sequence length so it broadcasts over the batch) and return the sum.

**Hint:**
* Make sure that the shapes of your positional encodings either match up with the shape of `x` or are broadcastable so that adding them is a valid operation.
* Inside `forward`, move `self.pos` to `x.device`.

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=1024):
        """
        max_len = max number of tokens in the sequence
        """
        # TODO: Implement the __init__ method
        # Don't forget to call super().__init__()!
        # Tests require this name:
        self.pos = ...

    def forward(self, x):
        """
        Input x is of shape (B, seq_len, d_embedding)
        """
        # TODO: Implement the forward method
        ...


In [ ]:
grader.check("q3f")

In [ ]:
# Sanity check the shape of the PositionalEncoding output (run after implementing PositionalEncoding.forward)
set_seed(SEED)
pe = PositionalEncoding(512, max_len=1024)

input = torch.rand((10, 12, 512))
print(f"Shape of input: {input.shape}")

pe_output = pe(input)

print(f"Shape of positional encoding output: {pe_output.shape}")

assert pe_output.shape == input.shape, f"Expected output shape to have the same shape as the input {input.shape}, but got {pe_output.shape}"


## Question 3g: Blocks are All You Need!

You've now implemented every building block of a decoder-only transformer: scaled dot-product attention, multi-head attention, the decoder layer, and positional encodings. Now you'll stack them into a complete model that maps token IDs to next-token predictions.

### TransformerDecoder Architecture
1. A token embedding layer, `nn.Embedding(vocab_size, d_model)`, which maps each token ID to a learned `d_model`-dimensional vector.
2. Your `PositionalEncoding` from Question 3f with `d_model` and `max_len`.
3. `num_layers` copies of your `DecoderLayer`, applied one after another.
4. A final LayerNorm. Pre-norm decoder layers leave their output unnormalized, so we normalize once before the output projection. The post-norm model in the paper doesn't need this step, since each of its layers already ends with a LayerNorm.
5. A linear layer from `d_model` to `vocab_size`. Next-token prediction is a classification problem with one class per word in the vocabulary, so this layer outputs one logit per word at every position.

**Task:** Implement the TransformerDecoder module.
1. Implement the `__init__` method.
    1. Instantiate **`self.embedding`** and **`self.positional_encoding`**.
    2. Instantiate **`self.layers`**, an `nn.ModuleList` of `num_layers` `DecoderLayer`s.
    3. Instantiate **`self.final_layernorm`** and **`self.linear`**.
2. Implement the `forward` method.
    1. Embed the token IDs and add positional encodings.
    2. Pass the result through each decoder layer in order.
    3. Apply the final LayerNorm, then the linear layer, and return the logits.

**Hints:**
* As with the attention heads in Question 3d, store the decoder layers in an `nn.ModuleList` so PyTorch registers their parameters.
* The input `x` holds integer token IDs, not embeddings.

**Expected behavior:**
* Input: `x` of shape `(batch_size, seq_len)` with integer token IDs
* Output: logits of shape `(batch_size, seq_len, vocab_size)`

In [ ]:
class TransformerDecoder(nn.Module):
    def __init__(
        self, vocab_size=1024, d_model=512, num_layers=6, num_heads=8, max_len=1024, verbose=False
    ):

        # TODO: Implement the __init__ method
        # Don't forget to call super().__init__()!
        # Tests require these names:
        self.embedding = ...
        self.positional_encoding = ...
        self.layers = ...
        self.final_layernorm = ...
        self.linear = ...

    def forward(self, x):

        # TODO: Implement the forward method
        # 1. Embeddings and positional encodings

        # 2. Decoder layers

        # 3. Final LayerNorm and linear layer
        return ...

In [ ]:
grader.check("q3g")

In [ ]:
# Shape progression through the stacked TransformerDecoder (small demo configuration)
set_seed(SEED)
B, seq, demo_vocab_size, d_model = 10, 100, 1024, 512
token_ids = torch.randint(0, demo_vocab_size, (B, seq))

model = TransformerDecoder(vocab_size=demo_vocab_size, d_model=d_model, num_layers=2, num_heads=8, verbose=True)
logits = model(token_ids)
print(f"TransformerDecoder output: {logits.shape}")
assert logits.shape == torch.Size([B, seq, demo_vocab_size]), f"Expected output shape (B, seq, vocab_size) {B, seq, demo_vocab_size}, but got {logits.shape}"

## Preparing a dataset to train the transformer

We'll use 1,000 short stories from TinyStories, a collection of synthetically generated stories. The file `data/tiny_stories_1000.jsonl` is the seed-189 shuffle of the TinyStories training split, so this cell does not download or shuffle the full dataset. The widget that shows loading progress may be broken or slow to appear. That is fine: the cell is done when it prints the dataset summary.

Attribution: [Eldan and Li, "TinyStories: How Small Can Language Models Be and Still Speak Coherent English?"](https://arxiv.org/abs/2305.07759). TinyStories is released under CDLA-Sharing-1.0.


In [ ]:
if IS_GRADING_ENV:
    class _TS:
        def shuffle(self, seed=None):
            return self
        def select(self, r):
            long = " ".join(["hello", "park", "cat", "dog"] * 40)
            return [{"text": long}] * 1000
    tiny_stories = _TS()
    tiny_stories_small = tiny_stories.shuffle(seed=SEED).select(range(1000))
else:
    tiny_stories_small = load_dataset(
        "json", data_files="data/tiny_stories_1000.jsonl", split="train"
    )
print(tiny_stories_small)


Let's check out one of the short stories in this dataset!


In [ ]:
print(tiny_stories_small[0]['text'])


We define a few helper functions to pre-process the text for transformer training.

First, we'll lowercase the stories. Then we **tokenize** them — split the text into chunks and map those chunks to integers the model can embed. Common choices include:
* Character-level tokenization: one token per character
* Byte Pair Encoding (BPE): merge frequent character pairs into subword units
* Word-level tokenization: one token per word
* Phrase tokenization: one token per common phrase

We'll keep it simple and use word-level tokens. The cell below builds a `word_to_id` dictionary over the unique words in the dataset, plus a few special tokens: `<start>`, `<pad>`, and `<end>` mark sequence boundaries, and `<unk>` stands in for any word that is not in the vocabulary.


In [ ]:
# Convert all the short stories to lowercase and collect them into a list
tiny_stories_text = [story['text'].lower() for story in tiny_stories_small]

# Extract all full words from text
def extract_full_words(text):
    """
    Given a string, extracts full words from the string
    Example:
        text = "The cat in the hat category"
        output = ['The', 'cat', 'in', 'the', 'hat', 'category']
    """
    pattern = r'\b\w+\b'
    return re.findall(pattern, text)

# Tokenize tiny story text - word level
paragraphs = []
for story in tiny_stories_text:
    story_paragraphs = story.split("\n\n") # split to remove new lines
    paragraphs.extend(story_paragraphs)

vocab = set()
for paragraph in paragraphs:
    full_words = extract_full_words(paragraph.lower())
    vocab.update(full_words)

# Sort the vocabulary so that every word gets the same ID each time this cell runs
# (iterating over a set directly gives a different order in every Python session)
word_to_id = {word: i for i, word in enumerate(sorted(vocab))}

PAD_TOKEN = '<pad>'
START_TOKEN = '<start>'
END_TOKEN = '<end>'
UNK_TOKEN = '<unk>'  # stands in for any word that is not in the vocabulary

word_to_id[PAD_TOKEN] = len(vocab)
word_to_id[START_TOKEN] = len(vocab) + 1
word_to_id[END_TOKEN] = len(vocab) + 2
word_to_id[UNK_TOKEN] = len(vocab) + 3

vocab.update([PAD_TOKEN, START_TOKEN, END_TOKEN, UNK_TOKEN])

print(f"Vocab size: {len(vocab)}")

print(f"Token ID of 'park': {word_to_id['park']}")
print(f"Token ID of 'cat': {word_to_id['cat']}")
print(f"Token ID of 'dog': {word_to_id['dog']}")


## Creating input and target sequences for next-token prediction

You might have noticed that our dataset earlier didn't come with any labels... but that's okay because we already have everything we need! We want to train the transformer to predict the next token: given everything up through the $i$-th token, how close does it get to the $i+1$-th token of the story?

The key idea is that for next-token prediction, the inputs and the targets are the same sequence shifted by 1 position.

Example:
* Sentence: "Dinosaur nuggets taste great with ketchup."
* If we pass in the token "with", the *target* is the actual next word: "ketchup"
* So each training pair is the same sequence shifted by one.

`create_sequences` below builds those pairs from TinyStories paragraphs.


In [ ]:
def create_sequences(
    stories,
    word_to_id,
    max_seq_length=20,
    pad_token=PAD_TOKEN,
    start_token=START_TOKEN,
    end_token=END_TOKEN,
):
    """Create sliding input/target windows for next-token prediction."""
    inputs = []
    targets = []

    paragraphs = []
    for story in stories:
        paragraphs.extend(story.split("\n\n"))

    for paragraph in paragraphs:
        words = extract_full_words(paragraph.lower())
        ids = (
            [word_to_id[start_token]]
            + [word_to_id[word] for word in words if word in word_to_id]
            + [word_to_id[end_token]]
        )
        if len(ids) <= max_seq_length:
            continue

        for i in range(len(ids) - max_seq_length):
            inputs.append(ids[i : i + max_seq_length])
            targets.append(ids[i + 1 : i + max_seq_length + 1])

    return (
        torch.tensor(inputs, dtype=torch.long),
        torch.tensor(targets, dtype=torch.long),
    )


max_seq_length = 20
inputs, targets = create_sequences(tiny_stories_text, word_to_id, max_seq_length)
print(f"Total sequences created: {len(inputs)}")
print(f"Each input sequence length: {len(inputs[0])}")
print(f"Each target sequence length: {len(targets[0])}")


### Training and validation DataLoaders

The next cell splits those sequences and wraps them in `TensorDataset` / `DataLoader` objects, the same pattern as Mini ImageNet earlier.


In [ ]:
split_idx = int(0.8 * len(inputs))
train_inputs, val_inputs = inputs[:split_idx], inputs[split_idx:]
train_targets, val_targets = targets[:split_idx], targets[split_idx:]

train_dataset = TensorDataset(train_inputs[:2000], train_targets[:2000])
val_dataset = TensorDataset(val_inputs[:200], val_targets[:200])

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print(f"Training sequences: {len(train_dataset)}")
print(f"Validation sequences: {len(val_dataset)}")
print(f"Training batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")

train_input, train_target = next(iter(train_loader))
print(f"First 10 input tokens: {train_input[0][:10]}")
print(f"First 10 target tokens: {train_target[0][:10]}")


## Question 3h: Training your Transformer

Using the transformer training loop provided below, train your transformer. Store the returned histories in `transformer_train_losses` and `transformer_val_losses`; you will plot them in the next part.

**Task**:
1. Instantiate a `TransformerDecoder` model.
2. Use `AdamW` as the optimizer.
    1. [PyTorch documentation for AdamW](https://docs.pytorch.org/docs/stable/generated/torch.optim.AdamW.html)
    2. We recommend trying learning rates between 0.00005 (`5e-5`) and 0.0002 (`2e-4`) to find which one the model learns best with.
3. Use `CrossEntropyLoss` as our criterion.
    1. [PyTorch documentation for CrossEntropyLoss](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html)
4. Try training your Transformer for different numbers of epochs! Based on the training loss curve we plot in the next question, and on the sample short stories generated in question 3i, pick a value for `num_epochs` that results in a model which generates the most understandable and coherent short stories.
    1. We recommend trying some values between 5 - 20 epochs.
    2. Your training loss should reach around **0.3**.
    3. For reference, the staff implementation took about 10 seconds per epoch on Google Colab's T4 and 5 seconds per epoch on Databricks' A10.

**Notes:**
* This training set is only a couple thousand short windows, so the model can memorize them. Once it does, the training loss keeps falling while the validation loss levels off or rises. This is what happens with a small dataset, and it doesn't mean the training loop is wrong.

**Hints:**
* Make sure you are passing the right `vocab_size` to your `TransformerDecoder`. The default value is **not** the size of the TinyStories vocabulary we built above, and a mismatch will cause an indexing error on the first batch.
* Don't forget to pass the parameters of your transformer model into the optimizer.
* Make sure that you turn on a GPU runtime before training!

In [ ]:
# Transformer training loop
def train_transformer(
    model,
    device,
    optimizer,
    criterion,
    num_epochs,
    train_dataloader,
    val_dataloader,
    vocab,
):
    if IS_GRADING_ENV:
        zeros = [0.0] * num_epochs
        return zeros, zeros

    assert str(device) in ["cuda", "mps"], "Please make sure you are using an accelerator (e.g. GPU or MPS) before training"

    # === SETUP ===
    # Move model to device
    model.to(device)

    # Lists to store metrics across epochs
    train_losses = []
    val_losses = []

    # === EPOCH LOOP ===
    for epoch in tqdm(range(num_epochs)):
        # === TRAINING PHASE ===
        model.train()  # Set model to training mode

        # Initialize metrics for this epoch
        total_train_loss = 0.0

        # === INNER LOOP (iterate over training batches) ===
        for batch_inputs, batch_targets in tqdm(train_dataloader, desc="Training"):
            # Move inputs and targets to device
            batch_inputs = batch_inputs.to(device)
            batch_targets = batch_targets.to(device)

            # Reset the gradients
            optimizer.zero_grad()

            # Forward pass
            logits = model(batch_inputs)  # [batch_size, seq_len, vocab_size]

            # Reshape for loss calculation
            loss = criterion(logits.reshape(-1, len(vocab)), batch_targets.reshape(-1))
            # Compute loss
            loss.backward()

            # Optional: Add gradient clipping to prevent exploding gradients
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            # Update the model parameters
            optimizer.step()

            # Accumulate the loss
            total_train_loss += loss.item()

        # === END OF INNER LOOP ===

        # Compute the average loss for this epoch
        avg_train_loss = total_train_loss / len(train_dataloader)
        train_losses.append(avg_train_loss)

        # === END OF TRAINING PHASE ===

        # === VALIDATION PHASE ===
        model.eval()  # Set model to evaluation mode
        total_val_loss = 0.0

        with torch.no_grad():
            # === INNER LOOP (iterate over validation batches) ===
            for batch_inputs, batch_targets in tqdm(val_dataloader, desc="Validation"):
                # Move inputs and targets to device
                batch_inputs = batch_inputs.to(device)
                batch_targets = batch_targets.to(device)

                # Forward pass
                logits = model(batch_inputs)

                # Compute loss
                loss = criterion(
                    logits.reshape(-1, len(vocab)), batch_targets.reshape(-1)
                )

                # Accumulate the loss
                total_val_loss += loss.item()
        # === END OF INNER LOOP ===

        # Compute the average loss for this epoch
        avg_val_loss = total_val_loss / len(val_dataloader)
        val_losses.append(avg_val_loss)

        # === END OF VALIDATION PHASE ===

        # Print epoch results
        print(f"\nEpoch {epoch + 1} Results:")
        print(f"  Training Loss: {avg_train_loss:.4f}")
        print(f"  Validation Loss: {avg_val_loss:.4f}")

    # === END OF EPOCH LOOP ===

    return train_losses, val_losses


In [ ]:
# TODO: Instantiate a TransformerDecoder model
transformer = ...

# TODO: Set the optimizer
optimizer = ...


# TODO: Set the loss criterion
criterion = ...

# TODO: Set the number of epochs
num_epochs = ...

# TODO: Train the model. Keep these variable names; the check below reads them.
transformer_train_losses, transformer_val_losses = ...

In [ ]:
grader.check("q3hi")

The autograder check above only checks that `transformer_train_losses` and `transformer_val_losses` are non-empty lists of the same length. To grade your model's performance, we'll plot and save the training curves. The cell below calls the provided `plot_metrics` helper on `transformer_train_losses` and `transformer_val_losses`, and saves the figure for grading.


<!-- BEGIN QUESTION -->



In [ ]:
fig_q3hii = plot_metrics(
    transformer_train_losses, transformer_val_losses,
    num_epochs=num_epochs, title="Transformer Training Metrics"
)
if not IS_GRADING_ENV:
    fig_q3hii.show()
save_figure(fig_q3hii, "q3hii")

if transformer_train_losses:
    print(f"Final Training Loss: {transformer_train_losses[-1]:.4f}")
if transformer_val_losses:
    print(f"Final Validation Loss: {transformer_val_losses[-1]:.4f}")


<!-- END QUESTION -->

## Question 3i: Transformer Inference

Our tiny transformer should now be ready to produce some of its own tiny stories! We've defined a function below that can be used to autoregressively generate new tokens from the transformer given a starting prompt. Feel free to play around with the `max_seq_len`, `max_length`, and `prompt` to see what stories you can generate with your transformer! The transformer may not be able to produce fully fluent sentences yet, but hopefully you see some coherence in the new stories it outputs.

**Note:** The transformer's final output is a vector of **logits** over the vocabulary (one score per word, just like the CNN's logits in Question 1b). We can get the ID of the next token it predicts by applying an `argmax` over the output vector at the last position in the sequence. To map this token ID back into a word, we can "flip" our dictionary which mapped words to ids into one that maps ids to words. Then, we can add this newly generated word to our tiny story so far, and keep sampling the next token from the transformer, ad infinitum! Any word in your prompt that is not in the vocabulary is replaced by the `<unk>` token.


In [ ]:
def generate_sample(model, word_to_id, id_to_word, max_seq_len=20, prompt="once upon a time", max_length=50):
    """
    Autoregressively generate a sample from the transformer

    Args:
        model (TransformerDecoder): The transformer model
        word_to_id (dict): Dictionary mapping words to their token IDs
        id_to_word (dict): Dictionary mapping token IDs to words
        max_seq_len (int): Maximum sequence length used when training the transformer
        prompt (str): The starting prompt for the generated story
        max_length (int): Maximum length of the generated story

    Returns:
        str: The generated story
    """
    model.eval()
    with torch.no_grad():
        tokens = [word_to_id[START_TOKEN]] + [word_to_id.get(word, word_to_id[UNK_TOKEN]) for word in prompt.split()]

        for _ in range(max_length):
            if len(tokens) >= max_seq_len:  # max sequence length used when training the transformer
                input_seq = torch.tensor([tokens[-1 * max_seq_len:]], dtype=torch.long, device=device)
            else:
                input_seq = torch.tensor([tokens], dtype=torch.long, device=device)

            logits = model(input_seq)
            next_token = torch.argmax(logits[0, -1]).item()
            tokens.append(next_token)

            if next_token == word_to_id.get(END_TOKEN, -1):
                break

        return ' '.join([id_to_word.get(token, UNK_TOKEN) for token in tokens])

id_to_word = {v: k for k, v in word_to_id.items()}

In [ ]:
# TODO: Generate your own tiny story!

sample_text = ...
print(f"Sample generation: {sample_text}")


In [ ]:
grader.check("q3i")

# Submission Checklist

Please make sure you have completed the following steps before submitting your assignment.

## Have you...
* Saved figures for the manual questions under `written/` (`q1bii.png`, `q2cii.png`, `q3hii.png`)?
* Run the export cell in the notebook to generate a zip file containing your code and `written/` files?
* Included an acknowledgement of any GenAI assistance you used while completing this assignment, if applicable? (See the [CS 189/289 syllabus](https://eecs189.org/fa26/syllabus/#collaboration-policy-and-academic-honesty) for full details of the course's GenAI policy and for an example acknowledgement format you can use).

## Submissions
* Submitted the exported zip to the `HW 4 Coding Part 1` assignment on Pensive?
    * Download the zip directly from Google Drive.
* Checked the PDF auto-submitted by the autograder to the `HW 4 Coding Part 1 PDF` assignment on Pensive?

Congratulations! You have completed Homework 4 Part 1!


# Before you submit

Save the notebook, run every cell that produces a written answer or figure, and then run the check below. Missing files under `written/` print a warning but do not fail the cell.


In [ ]:
warn_missing_written_files()


---

To double-check your work, the cell below will rerun all of the autograder tests.

In [ ]:
grader.check_all()

## Submission

The cell below will generate a zip file for you to submit.

Both this final submission generation cell and the autograder expect the assignment notebook to have the same name as the input to `otter.Notebook(...)` in the Otter initialization cell at the top of the notebook.
Do not rename your notebook, otherwise the final export and the autograder may run into errors processing your submission!

It will also include all completed responses to manually graded questions saved under `written/` and other required files. **Please save before exporting!**

Then, make sure to submit the generated zip file to the coding assignment on
Pensive. Pensive is set up to automatically format the files inside the zip
file's written/ directory into a PDF and submit it to the corresponding coding
PDF assignment automatically. After the autograder finishes running, please
double-check the submission that it automatically generates for the coding PDF assignment.


Do not add `run_tests=True` to `grader.export`. That option re-runs this entire
notebook in a fresh process, which retrains every model in the assignment from
scratch and will take a long time. It can also cause a crash by running out of GPU
memory, which will cause you to fail test cases that you previously passed and will
pass on Pensive. Use the `grader.check(...)` cells to check your work locally.
**Make sure you save the notebook before running the export cell.**


In [ ]:
# Save your notebook first, then run this cell to export your submission.
grader.export(pdf=False, files=['written'])